# Etapa 3 — Automação do Processo

Automação do processo de **extração, transformação e exportação de dados** do e-commerce fictício, garantindo reprodutibilidade e integração contínua dos resultados.


### 📦 Importação das bibliotecas

Nesta etapa, são importadas as bibliotecas necessárias para manipulação de dados (`pandas`), interação com banco SQLite (`sqlite3`), controle de diretórios (`os`, `shutil`) e registro de tempo (`datetime`).

In [7]:
# Célula 1 — Importações das bibliotecas
from google.colab import files

# Caminho e nome do script final
script_path = "automacao_ecommerce.py"

# Importando as bibliotecas principais
import pandas as pd
import sqlite3
import os
from datetime import datetime
import shutil

### ⚙️ Configurações iniciais

Definição dos caminhos principais do projeto — o banco de dados relacional (`SQLite`) e a pasta de exportação, que armazenará os relatórios gerados pela automação.

In [13]:
# Caminho do banco de dados e pasta de exportação
DB_PATH = 'ecommerce_realista.db'
EXPORT_PATH = 'data_export'

# Cria a pasta de exportação se não existir
os.makedirs(EXPORT_PATH, exist_ok=True)

# Verifica se o banco de dados existe localmente
if not os.path.exists(DB_PATH):
    print("⚠️ Banco de dados 'ecommerce_realista.db' não encontrado no ambiente.")
    print("📤 Faça o upload do arquivo para continuar.")
    uploaded = files.upload()  # solicita upload no Colab
    if 'ecommerce_realista.db' not in uploaded:
        raise FileNotFoundError("❌ O arquivo 'ecommerce_realista.db' é necessário para continuar.")

print("✅ Banco de dados encontrado e pronto para uso!")


⚠️ Banco de dados 'ecommerce_realista.db' não encontrado no ambiente.
📤 Faça o upload do arquivo para continuar.


Saving ecommerce_realista.db to ecommerce_realista.db
✅ Banco de dados encontrado e pronto para uso!


### 🔗 Conexão com o Banco de Dados

Realiza a conexão com o banco `SQLite` e carrega as tabelas principais do modelo estrela: `clientes`, `produtos` e `vendas`.  
Em caso de erro, o processo registra a mensagem sem interromper a execução.

In [14]:
# Conectando ao banco de dados e carregando as tabelas
print("Conectando ao banco de dados...")

try:
    with sqlite3.connect(DB_PATH) as conn:
        clientes = pd.read_sql("SELECT * FROM clientes", conn)
        produtos = pd.read_sql("SELECT * FROM produtos", conn)
        vendas = pd.read_sql("SELECT * FROM vendas", conn)
        print("✅ Tabelas carregadas com sucesso!\n")

except Exception as e:
    print(f"⚠️ Erro ao carregar tabelas: {e}")

Conectando ao banco de dados...
✅ Tabelas carregadas com sucesso!



## 🧩 Consolidação dos Dados

Nesta etapa, as tabelas **vendas**, **produtos** e **clientes** são unificadas em um único DataFrame principal.  
Além disso, é criada uma coluna de período (`ano_mes`), que permite análises temporais de faturamento e comportamento de vendas.

In [15]:
# Junta vendas → produtos → clientes
df = (
    vendas.merge(produtos, on='id_produto', how='left')
          .merge(clientes, on='id_cliente', how='left')
)

# Converte a coluna de data e gera o período em formato ano-mês
df['data_venda'] = pd.to_datetime(df['data_venda'])
df['ano_mes'] = df['data_venda'].dt.to_period('M')

print("🧩 Dados consolidados com sucesso!\n")


🧩 Dados consolidados com sucesso!



## 📊 Criação de Análises Resumidas

Após consolidar os dados, são geradas tabelas e métricas de resumo.  
Essas análises permitem identificar **tendências de faturamento por categoria e cliente**, além de calcular o **ticket médio geral**, um dos principais indicadores de desempenho do e-commerce.

In [16]:
# 1️⃣ Faturamento por categoria
resumo_cat = df.groupby('categoria', as_index=False)['valor_total'].sum()
resumo_cat.rename(columns={'valor_total': 'faturamento_total'}, inplace=True)

# 2️⃣ Faturamento por cliente
resumo_cli = df.groupby('nome', as_index=False)['valor_total'].sum()
resumo_cli.rename(columns={'valor_total': 'faturamento_cliente'}, inplace=True)

# 3️⃣ Ticket médio geral
ticket_medio = df['valor_total'].mean()
print(f"💰 Ticket médio: R$ {ticket_medio:.2f}\n")


💰 Ticket médio: R$ 391.43



## 💾 Exportação Automatizada dos Arquivos

Com os dados consolidados e as análises resumidas, esta etapa realiza a **exportação automática** dos resultados em formato `.csv`.  
Os arquivos são nomeados com a data da execução, garantindo **versionamento temporal** e rastreabilidade.  

In [17]:
# Data de referência no formato AAAA-MM-DD
data_hoje = datetime.now().strftime('%Y-%m-%d')

# Exporta os arquivos .csv para a pasta definida
df.to_csv(f'{EXPORT_PATH}/vendas_detalhadas_{data_hoje}.csv', index=False, encoding='utf-8')
resumo_cat.to_csv(f'{EXPORT_PATH}/resumo_categorias_{data_hoje}.csv', index=False, encoding='utf-8')
resumo_cli.to_csv(f'{EXPORT_PATH}/resumo_clientes_{data_hoje}.csv', index=False, encoding='utf-8')

print("💾 Arquivos exportados com sucesso!\n")


💾 Arquivos exportados com sucesso!



## 🧾 Registro de Execução (Log)

Para monitorar as execuções do processo automatizado, é criado um **arquivo de log** que registra a data e hora da última atualização.  
Esse registro é essencial para acompanhamento de rotina e auditoria de processos automatizados.

In [18]:
# Registra a data da última atualização no arquivo de log
# --------------------------------------------------------------

with open(f'{EXPORT_PATH}/log_execucao.txt', 'a') as log:
    log.write(f'Atualização realizada em {data_hoje}\n')

print("🧾 Log de execução atualizado.\n")


🧾 Log de execução atualizado.



## ⚙️ Exportar Automação como Script Executável (.py)

Nesta etapa final, o notebook gera automaticamente um **script `.py`** contendo todo o processo de automação —  
desde a extração e consolidação até a exportação e o registro em log.

Isso permite **agendar a execução do processo fora do ambiente do Colab**, como em um servidor local,  
utilizando o **Agendador de Tarefas do Windows** ou **cron jobs** no Linux.

In [19]:
# Caminho e nome do script final
script_path = "automacao_ecommerce.py"

# Conteúdo completo do script
script_content = """
import pandas as pd
import sqlite3
import os
from datetime import datetime

# Caminhos principais
DB_PATH = 'ecommerce_realista.db'
EXPORT_PATH = 'data_export'
os.makedirs(EXPORT_PATH, exist_ok=True)

# Conexão com o banco de dados
print("Conectando ao banco de dados...")

with sqlite3.connect(DB_PATH) as conn:
    clientes = pd.read_sql("SELECT * FROM clientes", conn)
    produtos = pd.read_sql("SELECT * FROM produtos", conn)
    vendas = pd.read_sql("SELECT * FROM vendas", conn)

print("Tabelas carregadas com sucesso!\\n")

# Consolidação dos dados
df = (
    vendas.merge(produtos, on='id_produto', how='left')
          .merge(clientes, on='id_cliente', how='left')
)
df['data_venda'] = pd.to_datetime(df['data_venda'])
df['ano_mes'] = df['data_venda'].dt.to_period('M')
print("Dados consolidados com sucesso!\\n")

# Análises resumidas
resumo_cat = df.groupby('categoria', as_index=False)['valor_total'].sum()
resumo_cat.rename(columns={'valor_total': 'faturamento_total'}, inplace=True)

resumo_cli = df.groupby('nome', as_index=False)['valor_total'].sum()
resumo_cli.rename(columns={'valor_total': 'faturamento_cliente'}, inplace=True)

ticket_medio = df['valor_total'].mean()
print(f"Ticket médio: R$ {ticket_medio:.2f}\\n")

# Exportação automatizada
data_hoje = datetime.now().strftime('%Y-%m-%d')
df.to_csv(f'{EXPORT_PATH}/vendas_detalhadas_{data_hoje}.csv', index=False, encoding='utf-8')
resumo_cat.to_csv(f'{EXPORT_PATH}/resumo_categorias_{data_hoje}.csv', index=False, encoding='utf-8')
resumo_cli.to_csv(f'{EXPORT_PATH}/resumo_clientes_{data_hoje}.csv', index=False, encoding='utf-8')

print("Arquivos exportados com sucesso!\\n")

# Registro de execução
with open(f'{EXPORT_PATH}/log_execucao.txt', 'a') as log:
    log.write(f'Atualização realizada em {data_hoje}\\n')

print("Log de execução atualizado.\\n")
print("Processo de automação concluído com sucesso!\\n")
"""

# Cria e salva o arquivo .py
with open(script_path, "w", encoding="utf-8") as f:
    f.write(script_content)

# Faz o download automático do script gerado
files.download(script_path)
print("📁 Script 'automacao_ecommerce.py' gerado e baixado com sucesso!")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📁 Script 'automacao_ecommerce.py' gerado e baixado com sucesso!
